In [ ]:
import dask.dataframe as dd
import numpy as np
import shap
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV, train_test_split, KFold
from xgboost import XGBRegressor
import logging
import multiprocessing
import time
from joblib import Parallel, delayed
import dask
from dask.distributed import Client

In [ ]:
# Initialize Dask client (this starts a local cluster on your machine)
client = Client()

def train_model(X: np.ndarray, y: np.ndarray, n_splits=5, test_size=0.2):
    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # Set up XGBoost regressor
    regressor = XGBRegressor(objective="reg:squarederror")

    # Define the grid search parameters
    param_grid = {
        'learning_rate': [0.1, 0.25, 0.35, 0.5],
        'max_depth': [7, 8, 9, 10, 11, 12],
        'subsample': [0.2, 0.4, 0.6, 0.8, 1],
        'n_estimators': [100, 150, 200, 250, 300, 350, 400]
    }

    # Perform grid search with cross-validation
    grid_search = GridSearchCV(
        regressor, 
        param_grid, 
        cv=KFold(n_splits=n_splits, shuffle=True, random_state=42),
        scoring="neg_mean_squared_error",
        n_jobs=-1,    
        verbose=3  # Show progress updates
    )

    
    print(f"Starting GridSearchCV with {multiprocessing.cpu_count()} CPUs...")
    start_time = time.time()
    
    # Train the model using grid search
    grid_search.fit(X_train, y_train)

    end_time = time.time()
    print(f"GridSearchCV completed in {end_time - start_time:.2f} seconds.")

    # Get the best parameters from the grid search
    best_params = grid_search.best_params_
    print("Best hyperparameters found:", best_params)
    
    # Train the best model on the full training set
    best_regressor = grid_search.best_estimator_

    # Evaluate the final model on the test set
    final_predictions = best_regressor.predict(X_test)
    print("Out-of-sample predictions:", final_predictions)

    return best_regressor, X_train, X_test, y_train, y_test


In [ ]:

# logging.basicConfig(level=logging.DEBUG)

# Define file paths
file_paths = {
    'Qle_all_sites': '/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/Qle_all_sites.csv',
    'SMtop1m_all_sites': '/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/SMtop1m_all_sites.csv',
    'LAI_all_sites_parallel': '/srv/ccrc/LandAP/z5218916/script/PLUMBER2/LSM_VPD_PLUMBER2/txt/process1_output/LAI_all_sites_parallel.csv'
}

# Load the data using Dask
def load_data_dask(file_path, columns):
    return dd.read_csv(file_path, usecols=columns, na_values=[''])

# Use dask to load all data in parallel
columns = {
    'Qle_all_sites': ['time', 'site_name', 'model_CABLE', 'VPD', 'obs_Tair', 'obs_Precip', 'obs_SWdown'],
    'SMtop1m_all_sites': ['CABLE_SMtop1m'],
    'LAI_all_sites_parallel': ['obs_LAI']
}

# Use Dask for parallel data loading
data = [load_data_dask(file_paths[key], columns[key]) for key in file_paths]

# Compute the Dask DataFrames (this loads the data into memory)
var_input = data[0].compute()
var_input['CABLE_SMtop1m'] = data[1].compute()['CABLE_SMtop1m']
var_input['obs_LAI'] = data[2].compute()['obs_LAI']

# Select a site for a trial
site_name = 'AU-Tum'
var_select = var_input

# Extract features and target
feature_names = ['VPD', 'obs_Tair', 'obs_Precip', 'obs_SWdown', 'CABLE_SMtop1m', 'obs_LAI']
X = var_select[feature_names].values
y = var_select['model_CABLE'].values

# Train the model and get results
model, X_train, X_test, y_train, y_test = train_model(X, y)

# Shutdown Dask client after computation
client.close()
